In [ ]:
import requests, zipfile
from pathlib import Path
from tqdm import tqdm
import zipfile
from pathlib import Path
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.io import loadmat

# Exploratory Data Analysis

## Physionet 2017

In [ ]:
candidatos = [
    Path.cwd() / "datasets" / "training2017",
    Path.cwd() / ".." / ".." / "datasets" / "training2017",
]
DATA_DIR = next(
    (p.resolve() for p in candidatos if (p / "REFERENCE.csv").exists()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        f"Não encontrei datasets/training2017 a partir de {Path.cwd()}"
    )

# Cada linha associa um registo a uma classe.
labels = pd.read_csv(
    DATA_DIR / "REFERENCE.csv",
    header=None,
    names=["record", "label"],
)
labels["record"] = labels["record"].astype(str).str.strip()
labels["label"] = labels["label"].astype(str).str.strip()

# Os headers .hea indicam frequência de amostragem e número de amostras.
metadata = []
for record in labels["record"]:
    header = DATA_DIR / f"{record}.hea"
    if not header.exists():
        continue

    campos = header.read_text().splitlines()[0].split()
    metadata.append({
        "record": record,
        "n_signals": int(campos[1]),
        "fs_hz": float(campos[2]),
        "n_samples": int(campos[3]),
    })

metadata = pd.DataFrame(metadata)
data = labels.merge(
    metadata,
    on="record",
    how="left",
    validate="one_to_one",
)
data["duration_s"] = data["n_samples"] / data["fs_hz"]

print(f"Dataset: {DATA_DIR}")
print(f"Registos com labels: {len(labels):,}")
print(f"Ficheiros .mat: {len(list(DATA_DIR.glob('*.mat'))):,}")
print(f"Headers .hea lidos: {len(metadata):,}")
print(f"Registos sem metadados: {data['fs_hz'].isna().sum()}")
print(f"Labels duplicadas: {labels['record'].duplicated().sum()}")

# Distribuição das classes
nomes_classes = {
    "N": "Normal",
    "A": "Fibrilhação auricular",
    "O": "Outro ritmo",
    "~": "Ruído",
}
contagens = labels["label"].value_counts().reindex(["N", "A", "O", "~"])
resumo = pd.DataFrame({
    "registos": contagens,
    "percentagem": (100 * contagens / contagens.sum()).round(2),
})
resumo.index = resumo.index.map(nomes_classes)
display(resumo)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(resumo.index, resumo["registos"])
ax.set_title("Número de registos por classe")
ax.set_ylabel("Registos")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Duração dos registos
display(data.groupby("label")["duration_s"].describe().round(2))

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=data, x="label", y="duration_s", order=["N", "A", "O", "~"], ax=ax)
ax.set_xticks(range(4), [nomes_classes[c] for c in ["N", "A", "O", "~"]])
ax.set_title("Duração dos ECGs por classe")
ax.set_xlabel("Classe")
ax.set_ylabel("Duração (segundos)")
plt.tight_layout()
plt.show()

# Primeiro sinal de cada classe, mostrando os primeiros 10 segundos.
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)

for ax, classe in zip(axes, ["N", "A", "O", "~"]):
    record = labels.loc[labels["label"] == classe, "record"].iloc[0]
    sinal = loadmat(DATA_DIR / f"{record}.mat")["val"].squeeze()
    fs = data.loc[data["record"] == record, "fs_hz"].iloc[0]
    tempo = np.arange(len(sinal)) / fs

    ax.plot(tempo, sinal, linewidth=0.7)
    ax.set_xlim(0, 10)
    ax.set_title(f"{nomes_classes[classe]} — {record}")
    ax.set_ylabel("Valor no ficheiro")

axes[-1].set_xlabel("Tempo (segundos)")
plt.tight_layout()
plt.show()

In [ ]:
# Verificacao dos ficheiros e caracteristicas numericas dos sinais
classes = ["N", "A", "O", "~"]
mat_ids = {p.stem for p in DATA_DIR.glob("*.mat")}
hea_ids = {p.stem for p in DATA_DIR.glob("*.hea")}
label_ids = set(labels["record"])
print("Sem .mat:", len(label_ids - mat_ids))
print("Sem .hea:", len(label_ids - hea_ids))
print("Ficheiros .mat sem label:", len(mat_ids - label_ids))
print("Ficheiros .hea sem label:", len(hea_ids - label_ids))
print("Frequencias de amostragem (Hz):", sorted(data["fs_hz"].dropna().unique()))

# Ler cada ECG um de cada vez: nao guardamos todos os sinais em memoria.
features = []
for row in data.itertuples(index=False):
    sinal = np.asarray(loadmat(DATA_DIR / f"{row.record}.mat")["val"]).squeeze()
    sinal = sinal.astype(np.float64, copy=False)
    finitos = np.isfinite(sinal)
    s = sinal[finitos]
    if s.size == 0:
        features.append({"record": row.record, "actual_n_samples": len(sinal), "finite_pct": 0.0})
        continue
    features.append({
        "record": row.record,
        "actual_n_samples": len(sinal),
        "finite_pct": 100 * finitos.mean(),
        "mean": s.mean(),
        "std": s.std(),
        "rms": np.sqrt(np.mean(s ** 2)),
        "median_abs": np.median(np.abs(s)),
        "peak_to_peak": np.ptp(s),
        "diff_std": np.diff(s).std() if len(s) > 1 else 0.0,
        "zero_pct": 100 * np.count_nonzero(s == 0) / len(s),
    })

features = pd.DataFrame(features)
data = data.merge(features, on="record", how="left", validate="one_to_one")
data["length_matches_header"] = data["actual_n_samples"] == data["n_samples"]
print("\nComprimento do sinal coincide com o header:", int(data["length_matches_header"].sum()), "/", len(data))
print("Percentagem minima de amostras finitas:", round(data["finite_pct"].min(), 3))
print("Registos com pelo menos uma amostra nao finita:", int((data["finite_pct"] < 100).sum()))

# Comparar estatisticas de amplitude e variacao entre classes.
display(data.groupby("label")[["duration_s", "std", "rms", "peak_to_peak", "diff_std", "zero_pct"]].agg(["median", "min", "max"]).round(2))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, coluna, titulo in zip(
    axes.flat,
    ["std", "rms", "peak_to_peak", "diff_std"],
    ["Desvio-padrao", "RMS", "Amplitude pico a pico", "Desvio-padrao da diferenca"],
):
    sns.boxplot(data=data, x="label", y=coluna, order=classes, showfliers=False, ax=ax)
    ax.set_title(titulo)
    ax.set_xlabel("Classe")
    ax.set_ylabel("Unidades armazenadas no ficheiro")
    ax.set_yscale("log")
plt.tight_layout()
plt.show()

# Espectro de potencia de um exemplo por classe (sinais brutos, sem filtragem).
from scipy.signal import welch

fig, ax = plt.subplots(figsize=(10, 5))
for classe in classes:
    record = labels.loc[labels["label"] == classe, "record"].iloc[0]
    sinal = loadmat(DATA_DIR / f"{record}.mat")["val"].squeeze().astype(float)
    fs = float(data.loc[data["record"] == record, "fs_hz"].iloc[0])
    frequencias, potencia = welch(sinal, fs=fs, nperseg=min(1024, len(sinal)))
    ax.semilogy(frequencias, potencia, label=f"{nomes_classes[classe]} ({record})")
ax.set_xlim(0, 40)
ax.set_xlabel("Frequencia (Hz)")
ax.set_ylabel("Densidade espectral (unidades armazenadas^2/Hz)")
ax.set_title("Espectro de potencia: um exemplo por classe")
ax.legend()
plt.tight_layout()
plt.show()

print("\nNota: estas caracteristicas sao descritivas; diferencas entre classes nao demonstram causalidade nem desempenho de um classificador.")

## Interpretação e resumo do EDA

### Interpretação

O conjunto tem **8.528 registos** e quatro classes com desequilíbrio marcado. **N** (normal) representa 59,22% dos registos, enquanto **~** (sinal ruidoso/não classificável) representa apenas 3,33%; **A** (fibrilhação auricular) corresponde a 8,65% e **O** (outros ritmos) a 28,80%. Num classificador futuro, a accuracy isolada pode esconder falhas nas classes minoritárias. Convém reportar precision, recall e F1 por classe, macro-F1 e a matriz de confusão.

A verificação estrutural encontrou os 8.528 ficheiros `.mat` e `.hea` esperados, sem IDs duplicados ou ficheiros em falta. Todos os sinais têm frequência de amostragem de 300 Hz, comprimentos compatíveis com os headers e amostras finitas. Isto confirma consistência estrutural, mas não garante que todos os ECG estejam clinicamente corretos ou livres de artefactos.

A duração mediana é 30 segundos em cada classe, mas o intervalo global é aproximadamente **9,05–60,95 segundos**. Os comprimentos variáveis devem ser considerados antes de alimentar sinais a um modelo que espere entradas com tamanho fixo; uma opção é usar janelas temporais consistentes, definida sem olhar para os dados de teste.

Nas estatísticas dos valores brutos, a classe **~** tem desvio-padrão mediano de 394,27 e amplitude pico a pico mediana de 4.440, superiores aos valores medianos das outras classes (desvio-padrão entre 166,67 e 177,18; pico a pico entre 1.596,5 e 1.777). Isto descreve maior variabilidade/amplitude neste grupo, mas não identifica a causa nem constitui um critério clínico. Há sobreposição entre classes e estas métricas simples não devem ser interpretadas como separadores perfeitos.

O gráfico espectral mostra apenas um exemplo por classe e os sinais não foram filtrados. Serve para inspeção inicial, não para generalizar diferenças espectrais a toda a classe; para isso, seria necessário resumir as características espectrais em todos os registos e validar a análise de forma adequada.

### Tabela-resumo

| Aspeto | Resultado observado | Interpretação / implicação |
|---|---|---|
| Dimensão | 8.528 registos; 8.528 ficheiros `.mat` e 8.528 `.hea` | Correspondência completa entre labels, sinais e headers. |
| Integridade estrutural | 0 IDs duplicados ou ficheiros em falta; 0 amostras não finitas; todos os comprimentos coincidem com os headers | Boa consistência estrutural; não equivale a validação clínica da qualidade. |
| Amostragem | 300 Hz em todos os registos | Frequência uniforme no conjunto. |
| Classes | N: 5.050 (59,22%); A: 738 (8,65%); O: 2.456 (28,80%); ~: 284 (3,33%) | Desequilíbrio relevante; avaliar desempenho por classe, não apenas por accuracy. |
| Duração | Mediana de 30 s em cada classe; intervalo global aproximado de 9,05–60,95 s | Comprimentos variáveis; definir como segmentar/padronizar antes da modelação. |
| Amplitude e dispersão | Desvio-padrão mediano: N 177,18; A 166,67; O 174,00; ~ 394,27. Pico a pico mediano: N 1.777; A 1.596,5; O 1.771; ~ 4.440 | A classe ~ tem valores maiores nestas estatísticas descritivas; não implica diagnóstico nem separabilidade perfeita. |
| Espectro | Um exemplo bruto por classe, usando Welch | Visualização ilustrativa; não permite inferência sobre toda a população de cada classe. |

**Conclusão:** o dataset é estruturalmente consistente e tem amostragem uniforme, mas apresenta forte desequilíbrio entre classes e durações variáveis. Antes de treinar um modelo, deve definir-se o tratamento dos comprimentos, separar treino/validação/teste sem fuga de informação e reportar métricas que revelem o desempenho nas classes minoritárias.